In [3]:
import os
import numpy as np
import pandas as pd
from typing import Dict, List
import time
from dotenv import load_dotenv
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from financerag.task import *
from financerag.retrieval import DenseRetrieval
from langchain_text_splitters.base import TextSplitter
from financerag.common import get_query_and_retrieved_corpus_text, process_retrieval_df, get_final_result 
from sentence_transformers import CrossEncoder
from transformers import pipeline
pd.set_option('display.max_colwidth', 400)
import warnings
warnings.filterwarnings('ignore')


In [4]:
load_dotenv(".env")
GG_API_KEY = os.environ.get('GOOGLE_API_KEY')
PINECONE_API_KEY = os.environ.get('PINECONE_API_KEY')

In [5]:
def load_query(dataset_name : str, new_path_to_query = None) -> Dict[str, str]:
    query_df = pd.read_json(f"finance_dataset/{dataset_name.lower()}_queries.jsonl/queries.jsonl", lines = True)
    # Convert into Dict[str, str]
    query_dict = {row['_id'] : row['text'] for i, row in query_df.iterrows()}
    return query_dict

    
def load_corpus(dataset_name : str, new_path_to_corpus = None) -> Dict[str, str]:
    corpus_df = pd.read_json(f"finance_dataset/{dataset_name.lower()}_corpus.jsonl/corpus.jsonl", lines = True)
    # Convert into Dict[str, str]
    corpus_dict = {row['_id'] : row['text'] for i, row in corpus_df.iterrows()}
    return corpus_dict

In [6]:
# Set up vector database
import faiss
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_community.vectorstores import FAISS
embeddings = GoogleGenerativeAIEmbeddings(model = "models/text-embedding-004", request_options = {'timeout' : 1000})
index = faiss.IndexFlatL2(len(embeddings.embed_query("hello world")))

vector_store = FAISS(
    embedding_function=embeddings,
    index=index,
    docstore=InMemoryDocstore(),
    index_to_docstore_id={},
)

In [7]:
def get_vector_store():
    embeddings = GoogleGenerativeAIEmbeddings(model = "models/text-embedding-004")
    index = faiss.IndexFlatL2(len(embeddings.embed_query("hello world")))

    vector_store = FAISS(
        embedding_function=embeddings,
        index=index,
        docstore=InMemoryDocstore(),
        index_to_docstore_id={},
    )
    return vector_store

In [8]:
import torch # type: ignore
torch.__version__

'2.3.0.dev20240311'

In [9]:
if torch.backends.mps.is_available():
    mps_device = torch.device("mps")

device = 'mps' if torch.backends.mps.is_available() else 'cpu'
device

'mps'

In [10]:
# Get dataset name first
import os
dataset_names = []
for f in os.listdir('finance_dataset'):
    if f.endswith("tsv"):
       dataset_names.append(f.split('_')[0])
dataset_names  

['MultiHeirtt',
 'FinQA',
 'FinanceBench',
 'ConvFinQA',
 'FinQABench',
 'TATQA',
 'FinDER']

In [11]:
CHUNK_SIZE = 1200
CHUNK_OVERLAP = 300
text_splitter = RecursiveCharacterTextSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)

In [12]:
for dataset_name in dataset_names:
    task_variable = f"{dataset_name.lower()}_task"
    script_string = f"""
    # {dataset_name} Task
    print(f"{dataset_name} task")
    {task_variable} = {dataset_name}Task()
    {task_variable}.load()
    {task_variable}_max_len = np.max([len(text) for text in {task_variable}.corpus.values()])
    {task_variable}_top_k = ({task_variable}_max_len // (CHUNK_SIZE - CHUNK_OVERLAP) + 1) * 10 + 10
    print("Retrieve", {task_variable}_top_k, f"documents/query for {dataset_name} task")
    vector_store = get_vector_store()
    {task_variable}_retriever = DenseRetrieval(vector_store = vector_store, dataset_name = {task_variable}.metadata.dataset_name)
    {task_variable}_retriever.load_corpus_with_splitting(text_splitter = text_splitter, corpus = {task_variable}.corpus, saved_index = False, batch_size = 500)

    {task_variable}_result = {task_variable}.retrieve(retriever = {task_variable}_retriever, top_k = {task_variable}_top_k)
    {task_variable}_reranker = CrossEncoderReranker(queries = {task_variable}.queries, corpus = {task_variable}.corpus, reranker = model)
    {task_variable}_final_result = {task_variable}_reranker.rerank(retrieved_result = {task_variable}_result, top_k = 10)
    {task_variable}.save_retrieved_results({task_variable}_final_result, method_name = 'dense_retrieval_split_document_and_reranking')
    """
    print(script_string)


    # MultiHeirtt Task
    print(f"MultiHeirtt task")
    multiheirtt_task = MultiHeirttTask()
    multiheirtt_task.load()
    multiheirtt_task_max_len = np.max([len(text) for text in multiheirtt_task.corpus.values()])
    multiheirtt_task_top_k = (multiheirtt_task_max_len // (CHUNK_SIZE - CHUNK_OVERLAP) + 1) * 10 + 10
    print("Retrieve", multiheirtt_task_top_k, f"documents/query for MultiHeirtt task")
    vector_store = get_vector_store()
    multiheirtt_task_retriever = DenseRetrieval(vector_store = vector_store, dataset_name = multiheirtt_task.metadata.dataset_name)
    multiheirtt_task_retriever.load_corpus_with_splitting(text_splitter = text_splitter, corpus = multiheirtt_task.corpus, saved_index = False, batch_size = 500)

    multiheirtt_task_result = multiheirtt_task.retrieve(retriever = multiheirtt_task_retriever, top_k = multiheirtt_task_top_k)
    multiheirtt_task_reranker = CrossEncoderReranker(queries = multiheirtt_task.queries, corpus = multiheirtt_task.corpus, reranke

In [13]:
from financerag.retrieval import BM25, BM25_Retriever
from financerag.rerank import CrossEncoderReranker
from sentence_transformers import CrossEncoder
%autoreload

In [14]:
model = CrossEncoder("cross-encoder/ms-marco-MiniLM-L6-v2", device=device)

In [15]:

# FinanceBench Task
print(f"FinanceBench task")
financebench_task = FinanceBenchTask()
financebench_task.load()
financebench_task_max_len = np.max([len(text) for text in financebench_task.corpus.values()])
financebench_task_top_k = (financebench_task_max_len // (CHUNK_SIZE - CHUNK_OVERLAP) + 1) * 10 + 10
print("Retrieve", financebench_task_top_k, f"documents/query for FinanceBench task")
vector_store = get_vector_store()
financebench_task_retriever = DenseRetrieval(vector_store = vector_store, dataset_name = financebench_task.metadata.dataset_name)
financebench_task_retriever.load_corpus_with_splitting(text_splitter = text_splitter, corpus = financebench_task.corpus, saved_index = False, batch_size = 500)

financebench_task_result = financebench_task.retrieve(retriever = financebench_task_retriever, top_k = financebench_task_top_k)
financebench_task_reranker = CrossEncoderReranker(queries = financebench_task.queries, corpus = financebench_task.corpus, reranker = model)
financebench_task_final_result = financebench_task_reranker.rerank(retrieved_result = financebench_task_result, top_k = 10)
financebench_task.save_retrieved_results(financebench_task_final_result, method_name = 'dense_retrieval_split_document_and_reranking')


# ConvFinQA Task
print(f"ConvFinQA task")
convfinqa_task = ConvFinQATask()
convfinqa_task.load()
convfinqa_task_max_len = np.max([len(text) for text in convfinqa_task.corpus.values()])
convfinqa_task_top_k = (convfinqa_task_max_len // (CHUNK_SIZE - CHUNK_OVERLAP) + 1) * 10 + 10
print("Retrieve", convfinqa_task_top_k, f"documents/query for ConvFinQA task")
vector_store = get_vector_store()
convfinqa_task_retriever = DenseRetrieval(vector_store = vector_store, dataset_name = convfinqa_task.metadata.dataset_name)
convfinqa_task_retriever.load_corpus_with_splitting(text_splitter = text_splitter, corpus = convfinqa_task.corpus, saved_index = False, batch_size = 500)

convfinqa_task_result = convfinqa_task.retrieve(retriever = convfinqa_task_retriever, top_k = convfinqa_task_top_k)
convfinqa_task_reranker = CrossEncoderReranker(queries = convfinqa_task.queries, corpus = convfinqa_task.corpus, reranker = model)
convfinqa_task_final_result = convfinqa_task_reranker.rerank(retrieved_result = convfinqa_task_result, top_k = 10)
convfinqa_task.save_retrieved_results(convfinqa_task_final_result, method_name = 'dense_retrieval_split_document_and_reranking')


# FinQABench Task
print(f"FinQABench task")
finqabench_task = FinQABenchTask()
finqabench_task.load()
finqabench_task_max_len = np.max([len(text) for text in finqabench_task.corpus.values()])
finqabench_task_top_k = (finqabench_task_max_len // (CHUNK_SIZE - CHUNK_OVERLAP) + 1) * 10 + 10
print("Retrieve", finqabench_task_top_k, f"documents/query for FinQABench task")
vector_store = get_vector_store()
finqabench_task_retriever = DenseRetrieval(vector_store = vector_store, dataset_name = finqabench_task.metadata.dataset_name)
finqabench_task_retriever.load_corpus_with_splitting(text_splitter = text_splitter, corpus = finqabench_task.corpus, saved_index = False, batch_size = 500)

finqabench_task_result = finqabench_task.retrieve(retriever = finqabench_task_retriever, top_k = finqabench_task_top_k)
finqabench_task_reranker = CrossEncoderReranker(queries = finqabench_task.queries, corpus = finqabench_task.corpus, reranker = model)
finqabench_task_final_result = finqabench_task_reranker.rerank(retrieved_result = finqabench_task_result, top_k = 10)
finqabench_task.save_retrieved_results(finqabench_task_final_result, method_name = 'dense_retrieval_split_document_and_reranking')


# TATQA Task
print(f"TATQA task")
tatqa_task = TATQATask()
tatqa_task.load()
tatqa_task_max_len = np.max([len(text) for text in tatqa_task.corpus.values()])
tatqa_task_top_k = (tatqa_task_max_len // (CHUNK_SIZE - CHUNK_OVERLAP) + 1) * 10 + 10
print("Retrieve", tatqa_task_top_k, f"documents/query for TATQA task")
vector_store = get_vector_store()
tatqa_task_retriever = DenseRetrieval(vector_store = vector_store, dataset_name = tatqa_task.metadata.dataset_name)
tatqa_task_retriever.load_corpus_with_splitting(text_splitter = text_splitter, corpus = tatqa_task.corpus, saved_index = False, batch_size = 500)

tatqa_task_result = tatqa_task.retrieve(retriever = tatqa_task_retriever, top_k = tatqa_task_top_k)
tatqa_task_reranker = CrossEncoderReranker(queries = tatqa_task.queries, corpus = tatqa_task.corpus, reranker = model)
tatqa_task_final_result = tatqa_task_reranker.rerank(retrieved_result = tatqa_task_result, top_k = 10)
tatqa_task.save_retrieved_results(tatqa_task_final_result, method_name = 'dense_retrieval_split_document_and_reranking')


# FinDER Task
print(f"FinDER task")
finder_task = FinDERTask()
finder_task.load()
finder_task_max_len = np.max([len(text) for text in finder_task.corpus.values()])
finder_task_top_k = (finder_task_max_len // (CHUNK_SIZE - CHUNK_OVERLAP) + 1) * 10 + 10
print("Retrieve", finder_task_top_k, f"documents/query for FinDER task")
vector_store = get_vector_store()
finder_task_retriever = DenseRetrieval(vector_store = vector_store, dataset_name = finder_task.metadata.dataset_name)
finder_task_retriever.load_corpus_with_splitting(text_splitter = text_splitter, corpus = finder_task.corpus, saved_index = False, batch_size = 500)

finder_task_result = finder_task.retrieve(retriever = finder_task_retriever, top_k = finder_task_top_k)
finder_task_reranker = CrossEncoderReranker(queries = finder_task.queries, corpus = finder_task.corpus, reranker = model)
finder_task_final_result = finder_task_reranker.rerank(retrieved_result = finder_task_result, top_k = 10)
finder_task.save_retrieved_results(finder_task_final_result, method_name = 'dense_retrieval_split_document_and_reranking')

FinanceBench task
Retrieve 90 documents/query for FinanceBench task


Reranking: 100%|██████████| 150/150 [01:27<00:00,  1.71it/s]


Saved result successfully to ./financerag_result/dense_retrieval_split_document_and_reranking/financebench_result.csv!
ConvFinQA task
Retrieve 180 documents/query for ConvFinQA task


Reranking: 100%|██████████| 421/421 [06:44<00:00,  1.04it/s]


Saved result successfully to ./financerag_result/dense_retrieval_split_document_and_reranking/convfinqa_result.csv!
FinQABench task
Retrieve 130 documents/query for FinQABench task


...

Saved result successfully to ./financerag_result/dense_retrieval_split_document_and_reranking/finqabench_result.csv!
TATQA task
Retrieve 180 documents/query for TATQA task


Reranking: 100%|██████████| 1663/1663 [32:49<00:00,  1.18s/it]


Saved result successfully to ./financerag_result/dense_retrieval_split_document_and_reranking/tatqa_result.csv!
FinDER task
Retrieve 380 documents/query for FinDER task


Add to vectorDB:   3%|▎         | 1/32 [01:03<32:49, 63.53s/it]


GoogleGenerativeAIError: Error embedding content: 500 An internal error has occurred. Please retry or report in https://developers.generativeai.google/guide/troubleshooting

In [15]:
for dataset_name in dataset_names:
    task_variable = f"{dataset_name.lower()}_task"
    script_string = f"""
    # {dataset_name} Task
    {task_variable}_final_result = extract_result({task_variable}_result, k = 10)
    {task_variable}.save_retrieved_results({task_variable}_final_result, method_name = 'dense_retrieval_split_only')
    """
    print(script_string)


    # MultiHeirtt Task
    multiheirtt_task_final_result = extract_result(multiheirtt_task_result, k = 10)
    multiheirtt_task.save_retrieved_results(multiheirtt_task_final_result, method_name = 'dense_retrieval_split_only')
    

    # FinQA Task
    finqa_task_final_result = extract_result(finqa_task_result, k = 10)
    finqa_task.save_retrieved_results(finqa_task_final_result, method_name = 'dense_retrieval_split_only')
    

    # FinanceBench Task
    financebench_task_final_result = extract_result(financebench_task_result, k = 10)
    financebench_task.save_retrieved_results(financebench_task_final_result, method_name = 'dense_retrieval_split_only')
    

    # ConvFinQA Task
    convfinqa_task_final_result = extract_result(convfinqa_task_result, k = 10)
    convfinqa_task.save_retrieved_results(convfinqa_task_final_result, method_name = 'dense_retrieval_split_only')
    

    # FinQABench Task
    finqabench_task_final_result = extract_result(finqabench_task_result, k = 10)
  

In [23]:
def extract_result(task_result : Dict[str, Dict[str, float]], k = 10):
    final_result = {}
    for query_id, doc_dict in task_result.items():
        final_result[query_id] = {}
        for i, (corpus_id, score) in enumerate(doc_dict.items()):
            if (i == k): break
            final_result[query_id][corpus_id] = score
    return final_result

In [ ]:

# FinanceBench Task
financebench_task_final_result = extract_result(financebench_task_result, k = 10)
financebench_task.save_retrieved_results(financebench_task_final_result, method_name = 'dense_retrieval_split_only')


# ConvFinQA Task
convfinqa_task_final_result = extract_result(convfinqa_task_result, k = 10)
convfinqa_task.save_retrieved_results(convfinqa_task_final_result, method_name = 'dense_retrieval_split_only')


# FinQABench Task
finqabench_task_final_result = extract_result(finqabench_task_result, k = 10)
finqabench_task.save_retrieved_results(finqabench_task_final_result, method_name = 'dense_retrieval_split_only')


# TATQA Task
tatqa_task_final_result = extract_result(tatqa_task_result, k = 10)
tatqa_task.save_retrieved_results(tatqa_task_final_result, method_name = 'dense_retrieval_split_only')


# FinDER Task
finder_task_final_result = extract_result(finder_task_result, k = 10)
finder_task.save_retrieved_results(finder_task_final_result, method_name = 'dense_retrieval_split_only')

Saved result successfully to ./financerag_result/dense_retrieval_split_only/finqa_result.csv!


NameError: name 'financebench_task_result' is not defined

In [13]:





# TATQA Task
tatqa_task_final_result = extract_result(tatqa_task_result, k = 10)
tatqa_task.save_retrieved_results(tatqa_task_final_result, method_name = 'dense_retrieval_only')


# FinDER Task
finder_task_final_result = extract_result(finder_task_result, k = 10)
finder_task.save_retrieved_results(finder_task_final_result, method_name = 'dense_retrieval_only')

Saved result successfully to ./financerag_result/dense_retrieval_only/tatqa_result.csv!
Saved result successfully to ./financerag_result/dense_retrieval_only/finder_result.csv!


In [53]:
method_name = 'dense_retrieval_and_reranking'
final_result = get_final_result(dataset_names, method_name = method_name)

In [54]:
final_result

,query_id,corpus_id
0,q82d4c6ec,d81a04f9e
1,q82d4c6ec,d81a04fe4
2,q82d4c6ec,d8d3fbbaa
3,q82d4c6ec,d88be204e
4,q82d4c6ec,d8c22a02a
...,...,...
2155,q00218,BRK.A20231700
2156,q00218,BRK.A20231532
2157,q00218,BRK.A20232650
2158,q00218,BRK.A20230602


In [55]:
final_result.to_csv(f'submission_{method_name}.csv', index = False)

In [11]:
import torch.nn as nn

In [45]:

# FinQA Task
print(f"FinQA task")
finqa_task = FinQATask()
finqa_task.load()
vector_store = get_vector_store()
finqa_task_retriever = DenseRetrieval(vector_store = vector_store, dataset_name = finqa_task.metadata.dataset_name)
finqa_task_retriever.load_corpus_without_splitting(corpus = finqa_task.corpus, saved_index = False)
finqa_task_result = finqa_task.retrieve(retriever = finqa_task_retriever, top_k = 50)
finqa_task_reranker = CrossEncoderReranker(queries = finqa_task.queries, corpus = finqa_task.corpus, reranker = model)
finqa_task_final_result = finqa_task_reranker.rerank(finqa_task_result, top_k = 10)
finqa_task.save_retrieved_results(finqa_task_final_result, method_name = 'dense_retrieval_and_reranking')





FinQA task


Reranking: 100%|██████████| 1147/1147 [08:19<00:00,  2.29it/s]

Saved result successfully to ./financerag_result/dense_retrieval_and_reranking/finqa_result.csv!


In [46]:
# # MultiHeirtt Task
# multiheirtt_task_final_result = extract_result(multiheirtt_task_result, k = 10)
# multiheirtt_task.save_retrieved_results(multiheirtt_task_final_result, method_name = 'dense_retrieval_only')


# FinQA Task
finqa_task_final_result = extract_result(finqa_task_result, k = 10)
finqa_task.save_retrieved_results(finqa_task_final_result, method_name = 'dense_retrieval_only')





Saved result successfully to ./financerag_result/dense_retrieval_only/finqa_result.csv!


In [21]:
a = {'s': {'s' : 1}}
isinstance(a, Dict)

True